# Title
**Author**:  Sian Wilson <br>
**Date**:  21st October 2026 <br>
**Dataset Scope**: `dataset` <br>
**Report Type**: Ad-hoc analysis <br>

## Purpose

This notebook measures how often trains call at railway stations in Great Britain. It's the first step towards identifying "well-connected" stations for the NPPF work, potentially leading to a new dataset on the Planning Data platform.

It:

1. Downloads the full Network Rail timetable (the SCHEDULE feed in CIF format) from the [Network Rail Open Data service](https://publicdatafeeds.networkrail.co.uk/).
2. Filters the timetable to permanent (STP = `P`) schedules, and to stops with a public (GBTT) arrival or departure time.
3. Counts public arrivals and departures per hour at a chosen station on a chosen date.
4. Joins the stops to DfT [NaPTAN](https://naptan.api.dft.gov.uk/) rail station data to add station names and coordinates. Around 96% of stops match.

### Caveats

- **Base timetable only.** Short-term changes such as engineering works, overlays and cancellations (STP `O`/`N`/`C`) aren't applied, so the counts show the planned service rather than what actually runs on a given day.
- **The CIF full extract is only refreshed weekly**, on Fridays, so a download may be a few days old.
- **The data isn't trains only.** It includes some timetabled buses, rail replacement buses and ferries. These aren't filtered out yet.
- **Arrivals and departures can't simply be added together.** At a terminus such as Brighton, most arrivals and departures are the same train turning round.
- **Trams and metros are mostly missing.** Only services running on Network Rail track are included (e.g. Sheffield tram-train, parts of the Tyne & Wear Metro). Other tram networks would need TNDS or BODS data.
- **Some stops have no NaPTAN match**, mainly Heathrow bus stops, Overground/Underground platforms with their own TIPLOC, and Tyne & Wear Metro stops.



In [1]:
import os
import requests
import getpass
import csv
import gzip
import pandas as pd
from datetime import date

pd.set_option("display.max_rows", 100)

data_dir = "../../data/"
os.makedirs(data_dir, exist_ok=True)
cif_path = os.path.join(data_dir, "CIF_ALL_FULL_DAILY_toc-full.CIF.gz")
stops_path = os.path.join(data_dir, "out.csv")


## Data Import

In [13]:
## You must register for an account at https://publicdatafeeds.networkrail.co.uk/
## Once registered, run this cell and
## enter your Network Rail username and password

user = os.environ.get("NR_USER") or input("Network Rail email: ")
password = os.environ.get("NR_PASS") or getpass.getpass("Network Rail password: ")

In [14]:
# Import Network Rail data
r = requests.get(
    "https://publicdatafeeds.networkrail.co.uk/ntrod/CifFileAuthenticate",
    params={"type": "CIF_ALL_FULL_DAILY", "day": "toc-full.CIF.gz"},
    auth=(user, password),
    stream=True,
)
r.raise_for_status()
with open(cif_path, "wb") as f:
    for chunk in r.iter_content(1 << 20):
        f.write(chunk)
print(f"Saved {cif_path}")

Saved ../../data/CIF_ALL_FULL_DAILY_toc-full.CIF.gz


In [15]:
# Import NaPTAN station data
naptan = pd.read_csv(
    "https://naptan.api.dft.gov.uk/v1/access-nodes?atcoAreaCodes=910&dataFormat=csv",
    dtype=str,
)

## Analysis

In [16]:
def _public_time(s):
    s = s.strip()
    return "" if s in ("", "0000") else s  # 0000 / blank = no public time


def filter_cif(src, dst):
    """Write permanent (STP=P) schedules' public stops from a CIF file to a CSV.

    src can be the .CIF file or the downloaded .CIF.gz.
    """
    opener = gzip.open if src.endswith(".gz") else open
    locations = {}  # tiploc -> (crs, name)                                    # changed
    with opener(src, "rt", encoding="latin-1") as f, open(dst, "w", newline="") as out:
        w = csv.writer(out)
        w.writerow(["train_uid", "date_from", "date_to", "days_run", "headcode",
                    "record", "tiploc", "crs", "name", "public_arrival", "public_departure"])  # changed
        bs = None  # current BS record if it's STP=P, else None
        for line in f:
            rec = line[0:2]
            if rec == "TI":
                locations[line[2:9].strip()] = (line[53:56].strip(), line[18:44].strip().title())  # changed
            elif rec == "BS":
                bs = line if line[79:80] == "P" else None
            elif bs is None:
                continue
            elif rec in ("LO", "LI", "LT"):
                tiploc = line[2:9].strip()
                if rec == "LO":
                    arr, dep = "", _public_time(line[15:19])
                elif rec == "LI":
                    arr, dep = _public_time(line[25:29]), _public_time(line[29:33])
                else:  # LT
                    arr, dep = _public_time(line[15:19]), ""
                if arr or dep:
                    w.writerow([bs[3:9], bs[9:15], bs[15:21], bs[21:28], bs[32:36],
                                rec, tiploc, *locations.get(tiploc, ("", "")), arr, dep])  # changed
                if rec == "LT":
                    bs = None


In [17]:
# Parse the Network Rail data 
filter_cif(cif_path, stops_path)

In [18]:
# Read data in as pandas DataFrame for quick exploration
stops = pd.read_csv(stops_path, dtype=str)
stops.head()

,train_uid,date_from,date_to,days_run,headcode,record,tiploc,crs,name,public_arrival,public_departure
0,C00045,260517,261206,0000001,1T95,LO,PADTON,PAD,London Paddington,NaN,2255
1,C00045,260517,261206,0000001,1T95,LI,HTRWAPT,HXX,Heathrow Terminals 2 & 3,2314,2316
2,C00045,260517,261206,0000001,1T95,LT,HTRWTM5,HWV,Heathrow Terminal 5,2320,NaN
3,C00046,260517,261206,0000001,1T96,LO,PADTON,PAD,London Paddington,NaN,2312
4,C00046,260517,261206,0000001,1T96,LI,HTRWAPT,HXX,Heathrow Terminals 2 & 3,2329,2331


In [19]:
# Function to filter by TIPLOC code 
## Calculates how many trains per hour on a set day
def trains_per_hour(stops, tiploc, day):
    """Count public arrivals and departures per hour at a TIPLOC on one date."""
    day = date.fromisoformat(day)
    yymmdd = day.strftime("%y%m%d")
    at = stops[stops["tiploc"] == tiploc]
    # schedule must be valid on this date and run on this weekday (days_run is Mon..Sun)
    runs = (
        (at["date_from"] <= yymmdd)
        & (at["date_to"] >= yymmdd)
        & (at["days_run"].str[day.weekday()] == "1")
    )
    at = at[runs]
    counts = pd.DataFrame({
        "arrivals": at["public_arrival"].dropna().str[:2].value_counts(),
        "departures": at["public_departure"].dropna().str[:2].value_counts(),
    }).fillna(0).astype(int).sort_index()
    counts.index = counts.index + ":00"
    counts.index.name = "hour"
    return counts

# Example of Brighton on Wednesday 21st October 2026
## We've been advised to use Wednesday as a good example day
brighton = trains_per_hour(stops, "BRGHTN", "2026-10-21")
brighton

,arrivals,departures
hour,,
00:00,9,3
01:00,4,0
02:00,2,0
04:00,0,1
05:00,4,14
06:00,11,14
07:00,13,18
08:00,14,14
09:00,16,14


### Join to NaPTAN station data

In [20]:
# Check data
naptan.head()

,ATCOCode,NaptanCode,PlateCode,CleardownCode,CommonName,CommonNameLang,ShortCommonName,ShortCommonNameLang,Landmark,LandmarkLang,...,TimingStatus,DefaultWaitTime,Notes,NotesLang,AdministrativeAreaCode,CreationDateTime,ModificationDateTime,RevisionNumber,Modification,Status
0,9100ABDARE,NaN,NaN,NaN,Aberdare Rail Station,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,110,2006-09-08T14:30:00,2007-09-26T13:00:00,1,revise,active
1,9100ASHYDN,NaN,NaN,NaN,Ashley Down,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,110,2025-04-25T11:18:00,2025-04-25T11:18:00,0,new,active
2,9100SWNACFC,NaN,NaN,NaN,Corfe Castle,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,110,2025-09-11T11:18:00,2025-09-11T11:18:00,0,new,active
3,9100BEULYPK,NaN,NaN,NaN,Beaulieu Park,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,110,2025-08-01T09:19:00,2025-08-01T13:00:00,0,new,active
4,9100ELINTN,NaN,NaN,NaN,East Linton,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,110,2025-04-25T15:13:00,2025-04-25T15:13:00,0,new,active


In [21]:
# Merge with Network Rail data
stations = pd.DataFrame({
    "tiploc": naptan["ATCOCode"].str.removeprefix("9100"),
    "station_name": naptan["CommonName"].str.removesuffix(" Rail Station"),
    "latitude": naptan["Latitude"].astype(float),
    "longitude": naptan["Longitude"].astype(float),
})

stops_merged = stops.merge(stations, on="tiploc", how="left")

In [22]:
stops_merged.head()

,train_uid,date_from,date_to,days_run,headcode,record,tiploc,crs,name,public_arrival,public_departure,station_name,latitude,longitude
0,C00045,260517,261206,0000001,1T95,LO,PADTON,PAD,London Paddington,NaN,2255,London Paddington,51.515996,-0.176174
1,C00045,260517,261206,0000001,1T95,LI,HTRWAPT,HXX,Heathrow Terminals 2 & 3,2314,2316,Heathrow Terminals 2 & 3,51.471852,-0.454102
2,C00045,260517,261206,0000001,1T95,LT,HTRWTM5,HWV,Heathrow Terminal 5,2320,NaN,Heathrow Terminal 5,51.470053,-0.490589
3,C00046,260517,261206,0000001,1T96,LO,PADTON,PAD,London Paddington,NaN,2312,London Paddington,51.515996,-0.176174
4,C00046,260517,261206,0000001,1T96,LI,HTRWAPT,HXX,Heathrow Terminals 2 & 3,2329,2331,Heathrow Terminals 2 & 3,51.471852,-0.454102
